

Algorithms:
  - BFS  (uninformed, bonus: shows the MAP_B trap, minimizes steps not cost)
  - UCS  (uninformed, optimal cost)
  - A*   (informed, Manhattan heuristic, optimal cost)

HOW TO CHANGE THE MAP:
    Change the variable GRID below (e.g. GRID = MAP_B), then set RUN_ALL = False.
    You can also paste your own map as a list of strings.


## How to run
1. Run all cells from top to bottom (Runtime → Run all in Colab, or Run All in Jupyter).
   Or export as a script and run: `python task1_intelligent.py`
2. To run on ONE map, change these two variables in the "Maps" cell:
   - `GRID = MAP_B`   (choose MAP_A, MAP_B, MAP_C, or paste your own list of strings)
   - `RUN_ALL = False`  (if True, GRID is ignored and all 3 maps are run)
3. Your own map must contain exactly one `S` and one `G`,
   using `.` (cost 1), `~` (cost 3), `#` (wall).

In [ ]:
import heapq
import time
from collections import deque

# Maps
MAP_A = [
    "S..#....",
    "##.#.##.",
    "....~~..",
    ".####.#.",
    "...~..#G",
]
MAP_B = [
    "S~~~~~~~G",
    ".#######.",
    ".........",
    ".#.#.#.#.",
    ".........",
]
MAP_C = [
    "S..#....",
    ".#.#.##.",
    "...#..#.",
    ".###.##.",
    "....#..G",
]
#  CHANGE  MAP
GRID = MAP_A
RUN_ALL = True

# Problem

def find_start_goal(grid):
    """Scan the grid to find S and G. Returns (start, goal) as (row, col)."""
    start = goal = None
    for r, row in enumerate(grid):
        for c, ch in enumerate(row):
            if ch == "S":
                start = (r, c)
            elif ch == "G":
                goal = (r, c)
    return start, goal


def step_cost(grid, pos):
    """Cost of ENTERING a cell: '~' = 3, anything else walkable = 1 (G = 1)."""
    r, c = pos
    return 3 if grid[r][c] == "~" else 1


def neighbors(grid, pos):
    """Valid 4-directional neighbours (Up, Down, Left, Right), no walls."""
    r, c = pos
    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[nr]) and grid[nr][nc] != "#":
            yield (nr, nc)


def manhattan(a, b):
    """Heuristic: Manhattan distance. Admissible because min step cost is 1."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def reconstruct(parent, goal):
    """Walk back through the parent links to build the path S -> G."""
    path, node = [], goal
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]


def path_cost(grid, path):
    """Sum of entering costs along the path (S is not counted)."""
    return sum(step_cost(grid, p) for p in path[1:])


def make_result(name, grid, path, expanded, max_frontier, elapsed):
    return {
        "name": name,
        "path": path,
        "length": len(path) - 1 if path else None,
        "cost": path_cost(grid, path) if path else None,
        "expanded": expanded,
        "max_frontier": max_frontier,
        "time_ms": elapsed * 1000,
    }


def bfs(grid, start, goal):
    """Breadth-First Search: fewest steps, ignores cost."""
    t0 = time.perf_counter()
    frontier = deque([start])
    parent = {start: None}
    expanded, max_frontier = 0, 1
    while frontier:
        node = frontier.popleft()
        expanded += 1
        if node == goal:
            return make_result("BFS", grid, reconstruct(parent, goal),
                               expanded, max_frontier, time.perf_counter() - t0)
        for nb in neighbors(grid, node):
            if nb not in parent:
                parent[nb] = node
                frontier.append(nb)
        max_frontier = max(max_frontier, len(frontier))
    return make_result("BFS", grid, None, expanded, max_frontier,
                       time.perf_counter() - t0)


def ucs(grid, start, goal):
    """Uniform-Cost Search: expands the node with the lowest g(n)."""
    t0 = time.perf_counter()
    counter = 0
    frontier = [(0, counter, start)]
    parent = {start: None}
    best_g = {start: 0}
    closed = set()
    expanded, max_frontier = 0, 1
    while frontier:
        g, _, node = heapq.heappop(frontier)
        if node in closed:
            continue
        closed.add(node)
        expanded += 1
        if node == goal:
            return make_result("UCS", grid, reconstruct(parent, goal),
                               expanded, max_frontier, time.perf_counter() - t0)
        for nb in neighbors(grid, node):
            new_g = g + step_cost(grid, nb)
            if nb not in best_g or new_g < best_g[nb]:
                best_g[nb] = new_g
                parent[nb] = node
                counter += 1
                heapq.heappush(frontier, (new_g, counter, nb))
        max_frontier = max(max_frontier, len(frontier))
    return make_result("UCS", grid, None, expanded, max_frontier,
                       time.perf_counter() - t0)


def astar(grid, start, goal):
    """A*: expands the node with the lowest f(n) = g(n) + h(n)."""
    t0 = time.perf_counter()
    counter = 0
    frontier = [(manhattan(start, goal), counter, 0, start)]
    parent = {start: None}
    best_g = {start: 0}
    closed = set()
    expanded, max_frontier = 0, 1
    while frontier:
        _, _, g, node = heapq.heappop(frontier)
        if node in closed:
            continue
        closed.add(node)
        expanded += 1
        if node == goal:
            return make_result("A* (Manhattan)", grid, reconstruct(parent, goal),
                               expanded, max_frontier, time.perf_counter() - t0)
        for nb in neighbors(grid, node):
            new_g = g + step_cost(grid, nb)
            if nb not in best_g or new_g < best_g[nb]:
                best_g[nb] = new_g
                parent[nb] = node
                counter += 1
                heapq.heappush(frontier,
                               (new_g + manhattan(nb, goal), counter, new_g, nb))
        max_frontier = max(max_frontier, len(frontier))
    return make_result("A* (Manhattan)", grid, None, expanded, max_frontier,
                       time.perf_counter() - t0)


# Output
def show_path_on_map(grid, path):
    """Print the map with path cells marked by '*' (S and G are kept)."""
    cells = set(path) if path else set()
    for r, row in enumerate(grid):
        print("   " + "".join(
            "*" if (r, c) in cells and ch not in "SG" else ch
            for c, ch in enumerate(row)))


def report(grid, result):
    print(f"\n--- {result['name']} ---")
    if result["path"] is None:
        print("No path found")
        print(f"Nodes expanded: {result['expanded']}")
    else:
        print(f"Path: {result['path']}")
        print(f"Path length (moves): {result['length']}")
        print(f"Total cost: {result['cost']}")
        print(f"Nodes expanded: {result['expanded']}")
        show_path_on_map(grid, result["path"])
    print(f"Max frontier size: {result['max_frontier']}")
    print(f"Run time: {result['time_ms']:.4f} ms")


def run_on_map(map_name, grid):
    print("=" * 60)
    print(f"{map_name}")
    print("=" * 60)
    start, goal = find_start_goal(grid)
    print(f"Start = {start}, Goal = {goal}")
    results = []
    for algo in (bfs, ucs, astar):
        res = algo(grid, start, goal)
        report(grid, res)
        results.append(res)
    return results


def print_table(all_results):
    print("\n" + "=" * 86)
    print("SUMMARY TABLE")
    print("=" * 86)
    print(f"{'Map':<7}{'Algorithm':<17}{'Length':>8}{'Cost':>7}"
          f"{'Expanded':>10}{'MaxFrontier':>13}{'Time(ms)':>11}")
    print("-" * 86)
    for map_name, results in all_results:
        for r in results:
            length = r["length"] if r["length"] is not None else "-"
            cost = r["cost"] if r["cost"] is not None else "-"
            print(f"{map_name:<7}{r['name']:<17}{str(length):>8}{str(cost):>7}"
                  f"{r['expanded']:>10}{r['max_frontier']:>13}{r['time_ms']:>11.4f}")
        print("-" * 86)


In [ ]:
if __name__ == "__main__":
    if RUN_ALL:
        maps = [("MAP_A", MAP_A), ("MAP_B", MAP_B), ("MAP_C", MAP_C)]
    else:
        maps = [("GRID", GRID)]

    summary = [(name, run_on_map(name, g)) for name, g in maps]
    print_table(summary)

MAP_A
Start = (0, 0), Goal = (4, 7)

--- BFS ---
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Path length (moves): 11
Total cost: 15
Nodes expanded: 28
   S**#....
   ##*#.##.
   ..******
   .####.#*
   ...~..#G
Max frontier size: 5
Run time: 0.0450 ms

--- UCS ---
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Path length (moves): 11
Total cost: 15
Nodes expanded: 28
   S**#....
   ##*#.##.
   ..******
   .####.#*
   ...~..#G
Max frontier size: 4
Run time: 0.0485 ms

--- A* (Manhattan) ---
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Path length (moves): 11
Total cost: 15
Nodes expanded: 21
   S**#....
   ##*#.##.
   ..******
   .####.#*
   ...~..#G
Max frontier size: 5
Run time: 0.0388 ms
MAP_B
Start = (0, 0), Goal = (0, 8)

--- BFS ---
Path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8)]


In [ ]:
def compare(summary):
    """Compare UCS (uninformed) vs A* (informed) on every map and print a conclusion."""

    print("COMPARISON: UCS (uninformed) vs A* (informed)")
    for map_name, results in summary:
        r = {x["name"]: x for x in results}
        bfs_r, ucs_r, a_r = r["BFS"], r["UCS"], r["A* (Manhattan)"]
        print(f"\n{map_name}:")
        if ucs_r["path"] is None:
            print("  No solution exists. All algorithms report 'No path found'.")
            print(f"  Nodes expanded -> UCS: {ucs_r['expanded']}, A*: {a_r['expanded']}")
            continue
        # found the cheaper path?
        if ucs_r["cost"] == a_r["cost"]:
            print(f"  Cheaper path: tie (both cost {ucs_r['cost']}) -> both optimal.")
        else:
            best = "UCS" if ucs_r["cost"] < a_r["cost"] else "A*"
            print(f"  Cheaper path: {best}")
        #  expanded fewer nodes?
        if a_r["expanded"] < ucs_r["expanded"]:
            print(f"  Fewer nodes: A* ({a_r['expanded']} vs UCS {ucs_r['expanded']})")
        elif a_r["expanded"] > ucs_r["expanded"]:
            print(f"  Fewer nodes: UCS ({ucs_r['expanded']} vs A* {a_r['expanded']})")
        else:
            print(f"  Fewer nodes: tie ({a_r['expanded']})")
        #  BFS trap (shortest path is not the cheapest)
        if bfs_r["cost"] > ucs_r["cost"]:
            print(f"  Trap: BFS used fewer steps ({bfs_r['length']}) but cost more "
                  f"({bfs_r['cost']} vs {ucs_r['cost']}) -> BFS ignores rough floor.")

    print("\nConclusion:")
    print("  A* is the best choice for this robot. It finds the same optimal (cheapest)")
    print("  path as UCS because Manhattan distance is admissible (every move costs >= 1),")
    print("  but the heuristic guides it toward G, so it expands fewer nodes.")
    print("  BFS is not suitable: it minimizes steps, not cost, so it fails on rough floor (MAP_B).")
    print("  Trade-off: A* may keep a larger frontier (e.g., MAP_B) because it stores the heuristic-guided entries.")

compare(summary)

COMPARISON: UCS (uninformed) vs A* (informed)

MAP_A:
  Cheaper path: tie (both cost 15) -> both optimal.
  Fewer nodes: A* (21 vs UCS 28)

MAP_B:
  Cheaper path: tie (both cost 12) -> both optimal.
  Fewer nodes: A* (15 vs UCS 31)
  Trap: BFS used fewer steps (8) but cost more (22 vs 12) -> BFS ignores rough floor.

MAP_C:
  No solution exists. All algorithms report 'No path found'.
  Nodes expanded -> UCS: 13, A*: 13

Conclusion:
  A* is the best choice for this robot. It finds the same optimal (cheapest)
  path as UCS because Manhattan distance is admissible (every move costs >= 1),
  but the heuristic guides it toward G, so it expands fewer nodes.
  BFS is not suitable: it minimizes steps, not cost, so it fails on rough floor (MAP_B).
  Trade-off: A* may keep a larger frontier (e.g., MAP_B) because it stores the heuristic-guided entries.
